# The Dartboard Approach to RAG Retrieval

> **Goal:** Replace the flat top-K retrieval strategy with a *zone-weighted* approach that
> treats retrieved chunks differently based on how close they land to the "bullseye" of relevance.

**What you will learn in this notebook:**
1. Why standard top-K treats all results equally — and why that's sub-optimal.
2. The Dartboard metaphor: concentric rings of relevance.
3. How to classify retrieved chunks into Bullseye / Inner Ring / Outer Ring zones.
4. Fixed-threshold and dynamic-threshold implementations from scratch.
5. Zone-aware prompt construction that gives the LLM structured evidence.
6. Head-to-head comparison: flat top-K vs. dartboard on the same queries.
7. A complete RAG pipeline with dartboard retrieval and Qwen3-14B generation.
8. When (and when not) to use dartboard retrieval.

**Stack:** `sentence-transformers (bge-base-en-v1.5)`, `FAISS`, `transformers + bitsandbytes (Qwen3-14B 4-bit)`, `numpy`.  
No LangChain, no LlamaIndex, no OpenAI API — everything runs locally.

## 1 — The One-Size-Fits-All Problem

Standard top-K retrieval works like this:

```
query → embed → find K nearest neighbours → concatenate them → send to LLM
```

Every chunk gets the same treatment. Chunk #1 (similarity 0.92) and chunk #5 (similarity 0.61)
occupy equal real estate in the prompt. This creates three problems:

| Problem | Consequence |
|---|---|
| **Dilution** | Marginally relevant chunks water down the strong evidence |
| **Distraction** | The LLM may latch onto tangential information |
| **Wasted context window** | Low-relevance chunks consume tokens that could hold better material |

Imagine searching for *"What causes seasonal allergies?"* and retrieving:
- Chunk A (0.93): *"Pollen from trees, grasses, and weeds triggers IgE-mediated histamine release…"*
- Chunk B (0.72): *"The immune system distinguishes self from non-self using MHC molecules…"*
- Chunk C (0.58): *"Antihistamines were first synthesised in 1937 by Daniel Bovet…"*

Chunk A is a **bullseye**. Chunk B is useful background. Chunk C is trivia.
Flat top-K jams all three together with equal weight. The dartboard approach says:
*treat them differently.*

## 2 — The Dartboard Metaphor

Picture a dartboard. The query is the centre — the exact question the user asked.
Every retrieved chunk "lands" somewhere on the board based on its similarity score:

```
              ┌─────────────────────────────┐
              │       OUTER  RING           │  sim < T_inner
              │   ┌─────────────────────┐   │
              │   │    INNER  RING      │   │  T_inner ≤ sim < T_bull
              │   │   ┌─────────────┐   │   │
              │   │   │  BULLSEYE   │   │   │  sim ≥ T_bull
              │   │   │   (query)   │   │   │
              │   │   └─────────────┘   │   │
              │   └─────────────────────┘   │
              └─────────────────────────────┘
```

| Zone | Meaning | Role in the prompt |
|---|---|---|
| **Bullseye** | Direct answers or highly relevant evidence | *Primary evidence* — appears first, gets the most emphasis |
| **Inner Ring** | Supporting context, definitions, related facts | *Supporting context* — fills in gaps |
| **Outer Ring** | Background, tangential, or low-confidence matches | *Background* — included only if space allows, clearly labelled |

The key insight: **each zone serves a different purpose**, so we *weight* and *position*
them differently in the prompt we send to the LLM.

## 3 — Precision vs. Recall Zones

The dartboard also encodes an **information-theoretic trade-off**:

- **Bullseye (inner zone) = high precision.** These chunks almost certainly contain the answer.
  If we *only* used bullseye chunks we would have very precise context, but we might miss
  important qualifiers, caveats, or related facts — low recall.

- **Outer Ring = high recall.** Including more chunks casts a wider net. We're less likely to
  miss relevant information, but we introduce noise — lower precision.

- **Inner Ring = the balanced zone.** It bridges precision and recall. These chunks support
  the bullseye without introducing too much noise.

By choosing how many tokens to allocate to each zone, you **tune the precision-recall dial**
without changing K. This is far more flexible than simply adjusting the number of
retrieved documents.

```
Precision ←──────────────────────────────────────→ Recall
  Bullseye only     Bullseye + Inner     All three zones
     (few tokens)        (moderate)        (many tokens)
```

## 4 — Environment Setup

In [ ]:
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

In [ ]:
import os, pathlib

# Google Drive cache for large models
CACHE = "/content/drive/MyDrive/models"
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    CACHE = str(pathlib.Path.home() / ".cache" / "huggingface" / "hub")

os.makedirs(CACHE, exist_ok=True)
os.environ["HF_HOME"]           = CACHE
os.environ["TRANSFORMERS_CACHE"] = CACHE
os.environ["HF_HUB_CACHE"]      = CACHE
print(f"Model cache → {CACHE}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

# ── Embedding model ──────────────────────────────────────────────
EMB_ID = "BAAI/bge-base-en-v1.5"
embedder = SentenceTransformer(EMB_ID, cache_folder=CACHE)
print(f"Embedder loaded: {EMB_ID}  dim={embedder.get_sentence_embedding_dimension()}")

# ── LLM: Qwen3-14B  4-bit NF4 ───────────────────────────────────
LLM_ID = "Qwen/Qwen3-14B"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(LLM_ID, cache_dir=CACHE)
model = AutoModelForCausalLM.from_pretrained(
    LLM_ID, quantization_config=bnb, device_map="auto", cache_dir=CACHE,
)
print(f"LLM loaded: {LLM_ID}  (4-bit NF4)")

In [ ]:
def generate(prompt, max_new_tokens=512, temperature=0.7):
    """Generate a response from Qwen3-14B with /no_think for deterministic output."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": prompt + "\n/no_think"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, top_p=0.9, do_sample=True,
        )
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Quick smoke test
print(generate("Say hello in one sentence.", max_new_tokens=30))

## 5 — Synthetic Knowledge Base

We build a **20-chunk corpus** about renewable energy. The chunks deliberately vary in
specificity — some are precise technical facts (bullseye-quality for specific queries),
some are broader context, and some are only tangentially related. This spread is what
makes the dartboard approach shine.

| Chunks | Topic |
|---|---|
| 0–4 | **Solar energy** — photovoltaic cells, efficiency, installations |
| 5–9 | **Wind energy** — turbines, offshore farms, capacity factors |
| 10–14 | **Battery storage** — lithium-ion, grid-scale, degradation |
| 15–19 | **Energy policy & economics** — subsidies, carbon pricing, grid integration |

In [ ]:
CHUNKS = [
    # ── Solar energy (0-4) ─────────────────────────────────────
    "Photovoltaic cells convert sunlight directly into electricity using the photovoltaic effect. When photons strike a semiconductor junction, they knock electrons free, creating a flow of direct current. Modern monocrystalline silicon cells achieve lab efficiencies of 26.7 percent, while commercial panels typically deliver 20-22 percent.",

    "The levelised cost of solar electricity (LCOE) fell 89 percent between 2010 and 2023, making utility-scale solar the cheapest source of new electricity in most regions. Key drivers include manufacturing scale in China, improved cell architectures like PERC and TOPCon, and lower balance-of-system costs.",

    "Perovskite solar cells are an emerging technology that can be deposited as thin films on flexible substrates. Tandem cells combining perovskite and silicon layers have reached 33.7 percent efficiency in the lab, potentially surpassing the Shockley-Queisser limit for single-junction devices.",

    "Solar irradiance varies with latitude, season, and cloud cover. The global horizontal irradiance in the Sahara Desert exceeds 2,500 kWh per square metre per year, while northern Europe receives roughly 1,000 kWh. Tracking systems that follow the sun can boost yield by 20-35 percent compared to fixed-tilt installations.",

    "Rooftop solar installations allow households and businesses to generate their own electricity, reducing transmission losses and peak demand on the grid. Net metering policies let prosumers sell excess power back to the utility, though the economic value of exported solar varies widely by jurisdiction.",

    # ── Wind energy (5-9) ──────────────────────────────────────
    "Horizontal-axis wind turbines extract kinetic energy from moving air by spinning a rotor connected to a generator. The power available scales with the cube of wind speed, so a doubling of wind speed means an eightfold increase in potential power output.",

    "Offshore wind farms benefit from stronger and more consistent winds than onshore sites. The average capacity factor for European offshore wind reached 42 percent in 2023, compared to 25-35 percent for onshore installations. Floating foundations are extending deployment to deeper waters.",

    "Wind turbine blades are typically made from fibreglass-reinforced epoxy and can exceed 100 metres in length. End-of-life blade disposal is an emerging environmental concern, as the thermoset composites are difficult to recycle. Research into thermoplastic resins and cement co-processing offers potential solutions.",

    "The capacity factor of a wind farm is the ratio of actual energy produced to the theoretical maximum if the turbines ran at rated power 24/7. It depends on wind resource, turbine technology, curtailment, and maintenance downtime. Top-performing sites achieve capacity factors above 50 percent.",

    "Wake effects occur when upstream turbines slow the wind for downstream machines. Computational fluid dynamics models help optimise turbine spacing within a farm to minimise wake losses, which can reduce total output by 10-20 percent if not managed.",

    # ── Battery storage (10-14) ────────────────────────────────
    "Lithium-ion batteries store energy through the intercalation of lithium ions between a graphite anode and a metal-oxide cathode. The most common cathode chemistries for grid storage are lithium iron phosphate (LFP) and nickel manganese cobalt (NMC), each offering different trade-offs in cost, energy density, and cycle life.",

    "Grid-scale battery storage systems smooth the variable output of solar and wind generators. A four-hour lithium-ion battery system can shift midday solar production to evening peak demand, reducing the need for natural gas peaker plants and lowering carbon emissions.",

    "Battery degradation occurs through side reactions that consume active lithium and increase cell impedance. Calendar ageing depends on temperature and state of charge, while cycle ageing depends on depth of discharge and charge rate. Typical LFP cells retain 80 percent capacity after 4,000-6,000 full cycles.",

    "Sodium-ion batteries are an emerging alternative that replaces lithium with abundant sodium. While their energy density is lower, they offer advantages in cost, low-temperature performance, and supply-chain security. CATL began mass production of sodium-ion cells in 2023.",

    "Solid-state batteries replace the liquid electrolyte with a solid ceramic or polymer conductor, potentially doubling energy density and eliminating flammability risks. Toyota and Samsung SDI have announced plans for commercial solid-state cells by 2027, though manufacturing scale-up remains challenging.",

    # ── Energy policy & economics (15-19) ──────────────────────
    "Feed-in tariffs guarantee renewable generators a fixed price per kilowatt-hour for a set contract period, typically 15-20 years. Germany's EEG feed-in tariff, introduced in 2000, was instrumental in scaling solar PV manufacturing globally by providing revenue certainty.",

    "Carbon pricing mechanisms, including emissions trading systems and carbon taxes, internalise the social cost of greenhouse gas emissions. The EU Emissions Trading System covers about 40 percent of EU emissions, with allowance prices exceeding 80 euros per tonne of CO2 in 2023.",

    "Grid integration of variable renewables requires investments in transmission, demand response, and storage. Curtailment — deliberately reducing renewable output when supply exceeds demand — reached 6 percent of total wind generation in Germany in 2023, signalling the need for more flexibility.",

    "Energy subsidies worldwide totalled 7 trillion dollars in 2022 according to the IMF, with the vast majority supporting fossil fuels through explicit subsidies and unpriced externalities. Redirecting even a fraction of these subsidies toward clean energy could dramatically accelerate the transition.",

    "The concept of a 'just transition' recognises that shifting away from fossil fuels affects workers and communities dependent on coal, oil, and gas industries. Effective transition policies include retraining programmes, early retirement schemes, and investment in new economic activities for affected regions.",
]

print(f"Knowledge base: {len(CHUNKS)} chunks")
for i, c in enumerate(CHUNKS):
    print(f"  [{i:>2}] {c[:75]}...")

## 6 — Build the FAISS Vector Index

We embed all chunks using `bge-base-en-v1.5` and store them in a FAISS index.
We use an Inner Product index since our embeddings are L2-normalised
(so dot product = cosine similarity).

In [ ]:
import numpy as np
import faiss

# Embed all chunks (normalised → dot product = cosine similarity)
chunk_embeddings = embedder.encode(CHUNKS, normalize_embeddings=True, show_progress_bar=True)
chunk_embeddings = np.array(chunk_embeddings, dtype="float32")

# Build FAISS index (inner product for cosine similarity on normalised vectors)
dim = chunk_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(chunk_embeddings)

print(f"FAISS index built: {index.ntotal} vectors, {dim} dimensions")
print(f"Embedding matrix shape: {chunk_embeddings.shape}")

## 7 — Standard Top-K Retrieval (Baseline)

First, let's build the baseline: plain top-K retrieval with no zone awareness.
Every chunk is treated equally.

In [ ]:
def retrieve_top_k(query, k=5):
    """Standard top-K retrieval. Returns (texts, scores) sorted by similarity."""
    q_emb = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({"chunk_id": int(idx), "score": float(score), "text": CHUNKS[idx]})
    return results

# Test it
query = "How efficient are solar panels?"
results = retrieve_top_k(query, k=7)
print(f"Query: '{query}'\n")
for r in results:
    print(f"  [{r['chunk_id']:>2}] sim={r['score']:.4f}  {r['text'][:70]}...")

## 8 — Visualising the Score Distribution

When you look at the similarity scores for a top-K retrieval, they're rarely uniform.
There's usually a **natural clustering**: a few high-relevance chunks, some moderate ones,
and a long tail of marginally relevant results. The dartboard approach exploits this
structure instead of ignoring it.

In [ ]:
# Retrieve a larger set to see the full score distribution
query = "How efficient are solar panels?"
all_results = retrieve_top_k(query, k=15)
scores = [r["score"] for r in all_results]

print(f"Query: '{query}'\n")
print("Score distribution across top-15 results:")
print(f"  Max:    {max(scores):.4f}")
print(f"  Min:    {min(scores):.4f}")
print(f"  Mean:   {np.mean(scores):.4f}")
print(f"  Median: {np.median(scores):.4f}")
print(f"  Std:    {np.std(scores):.4f}")
print()

# ASCII bar chart
print("Score histogram:")
for i, r in enumerate(all_results):
    bar_len = int((r["score"] - 0.3) * 80)  # scale for visibility
    bar = "█" * max(bar_len, 1)
    print(f"  [{r['chunk_id']:>2}] {r['score']:.4f} {bar}")

## 9 — Zone Classification: From Scores to Zones

The core of the dartboard approach is **classifying** each retrieved chunk into a zone.
The simplest method uses **fixed thresholds**:

```
if   sim ≥ T_bullseye  →  Bullseye     (direct answer)
elif sim ≥ T_inner     →  Inner Ring   (supporting context)
else                   →  Outer Ring   (background)
```

Choosing thresholds depends on your embedding model and domain. For `bge-base-en-v1.5`
with normalised embeddings (cosine similarity in [0, 1]), reasonable starting points are:
- **T_bullseye = 0.75** — only very close matches
- **T_inner = 0.55** — moderately related chunks

In [ ]:
def classify_zones_fixed(results, t_bull=0.75, t_inner=0.55):
    """Classify results into dartboard zones using fixed thresholds."""
    zones = {"bullseye": [], "inner_ring": [], "outer_ring": []}
    for r in results:
        r_with_zone = {**r}
        if r["score"] >= t_bull:
            r_with_zone["zone"] = "bullseye"
            zones["bullseye"].append(r_with_zone)
        elif r["score"] >= t_inner:
            r_with_zone["zone"] = "inner_ring"
            zones["inner_ring"].append(r_with_zone)
        else:
            r_with_zone["zone"] = "outer_ring"
            zones["outer_ring"].append(r_with_zone)
    return zones

# Classify our earlier results
query = "How efficient are solar panels?"
results = retrieve_top_k(query, k=10)
zones = classify_zones_fixed(results)

print(f"Query: '{query}'\n")
for zone_name, items in zones.items():
    print(f"  {zone_name.upper()} ({len(items)} chunks):")
    for item in items:
        print(f"    [{item['chunk_id']:>2}] sim={item['score']:.4f}  {item['text'][:60]}...")
    print()

## 10 — The Problem with Fixed Thresholds

Fixed thresholds are a good starting point, but they suffer from a critical flaw:
**the score distribution shifts with every query**.

- For a highly specific query (*"What is the Shockley-Queisser limit?"*), the top chunk
  might score 0.88 and the 5th chunk only 0.45. A threshold of 0.75 is reasonable.

- For a broad query (*"Tell me about energy"*), even the top chunk might only score 0.65.
  A 0.75 bullseye threshold would put *nothing* in the bullseye!

We need **dynamic thresholding** that adapts to each query's score landscape.

In [ ]:
# Demonstrate the score-distribution shift
queries = [
    "What is the Shockley-Queisser limit for solar cells?",
    "Tell me about energy",
    "How does lithium-ion battery degradation work?",
]

for q in queries:
    res = retrieve_top_k(q, k=7)
    scores = [r["score"] for r in res]
    z = classify_zones_fixed(res)
    b, i, o = len(z["bullseye"]), len(z["inner_ring"]), len(z["outer_ring"])
    print(f"Query: '{q}'")
    print(f"  Scores: max={max(scores):.3f}  min={min(scores):.3f}  spread={max(scores)-min(scores):.3f}")
    print(f"  Fixed zones → Bullseye={b}, Inner={i}, Outer={o}")
    print()

## 11 — Dynamic Thresholding

Instead of hard-coded numbers, we compute thresholds **relative to the score distribution**
of each query's results. Two popular strategies:

### Strategy A: Percentile-based
- Bullseye = top 20th percentile of retrieved scores
- Inner Ring = 20th–60th percentile
- Outer Ring = bottom 40 percent

### Strategy B: Standard-deviation-based
- Bullseye: score ≥ μ + σ
- Inner Ring: μ − 0.5σ ≤ score < μ + σ
- Outer Ring: score < μ − 0.5σ

Both strategies ensure *every query* gets a meaningful distribution across zones,
regardless of absolute score magnitudes.

In [ ]:
def classify_zones_percentile(results, bull_pct=80, inner_pct=40):
    """Classify using percentile-based dynamic thresholds."""
    scores = np.array([r["score"] for r in results])
    t_bull  = np.percentile(scores, bull_pct)
    t_inner = np.percentile(scores, inner_pct)
    
    zones = {"bullseye": [], "inner_ring": [], "outer_ring": []}
    for r in results:
        r_with_zone = {**r}
        if r["score"] >= t_bull:
            r_with_zone["zone"] = "bullseye"
            zones["bullseye"].append(r_with_zone)
        elif r["score"] >= t_inner:
            r_with_zone["zone"] = "inner_ring"
            zones["inner_ring"].append(r_with_zone)
        else:
            r_with_zone["zone"] = "outer_ring"
            zones["outer_ring"].append(r_with_zone)
    return zones, t_bull, t_inner


def classify_zones_stddev(results, bull_k=1.0, inner_k=-0.5):
    """Classify using mean ± k*std dynamic thresholds."""
    scores = np.array([r["score"] for r in results])
    mu, sigma = scores.mean(), scores.std()
    t_bull  = mu + bull_k * sigma
    t_inner = mu + inner_k * sigma
    
    zones = {"bullseye": [], "inner_ring": [], "outer_ring": []}
    for r in results:
        r_with_zone = {**r}
        if r["score"] >= t_bull:
            r_with_zone["zone"] = "bullseye"
            zones["bullseye"].append(r_with_zone)
        elif r["score"] >= t_inner:
            r_with_zone["zone"] = "inner_ring"
            zones["inner_ring"].append(r_with_zone)
        else:
            r_with_zone["zone"] = "outer_ring"
            zones["outer_ring"].append(r_with_zone)
    return zones, t_bull, t_inner

print("Dynamic thresholding functions defined.")

In [ ]:
# Compare fixed vs. dynamic thresholding on the three queries
for q in queries:
    res = retrieve_top_k(q, k=10)
    z_fixed = classify_zones_fixed(res)
    z_pct, t_b, t_i = classify_zones_percentile(res)
    z_std, t_b2, t_i2 = classify_zones_stddev(res)
    
    print(f"Query: '{q}'")
    print(f"  Fixed (0.75/0.55):       Bull={len(z_fixed['bullseye'])}  Inner={len(z_fixed['inner_ring'])}  Outer={len(z_fixed['outer_ring'])}")
    print(f"  Percentile (80/40):      Bull={len(z_pct['bullseye'])}  Inner={len(z_pct['inner_ring'])}  Outer={len(z_pct['outer_ring'])}   [T_bull={t_b:.3f}, T_inner={t_i:.3f}]")
    print(f"  StdDev (μ+σ / μ−0.5σ):  Bull={len(z_std['bullseye'])}  Inner={len(z_std['inner_ring'])}  Outer={len(z_std['outer_ring'])}   [T_bull={t_b2:.3f}, T_inner={t_i2:.3f}]")
    print()

## 12 — Zone-Aware Prompt Construction

Now comes the payoff. Instead of dumping all chunks into a single "Context:" block,
we structure the prompt with **explicit zone labels**:

```
PRIMARY EVIDENCE (highest relevance):
[bullseye chunks here]

SUPPORTING CONTEXT:
[inner ring chunks here]

BACKGROUND (may be tangential):
[outer ring chunks here]

QUESTION: ...
INSTRUCTION: Answer primarily from the PRIMARY EVIDENCE.
Use supporting context to fill gaps. Treat background
as potentially unreliable.
```

This gives the LLM a **structured signal** about which information to trust most.
It mirrors how human researchers treat their sources: primary literature first,
reviews second, Wikipedia last.

In [ ]:
def build_flat_prompt(query, results):
    """Standard flat prompt — all chunks treated equally."""
    context = "\n\n".join(f"[Chunk {r['chunk_id']}] {r['text']}" for r in results)
    return (
        f"Context:\n{context}\n\n"
        f"Question: {query}\n"
        f"Answer the question using only the context above. Be concise and specific."
    )


def build_dartboard_prompt(query, zones):
    """Zone-aware prompt with explicit evidence hierarchy."""
    sections = []
    
    if zones["bullseye"]:
        bull_text = "\n".join(f"  • {r['text']}" for r in zones["bullseye"])
        sections.append(f"PRIMARY EVIDENCE (highest relevance):\n{bull_text}")
    
    if zones["inner_ring"]:
        inner_text = "\n".join(f"  • {r['text']}" for r in zones["inner_ring"])
        sections.append(f"SUPPORTING CONTEXT:\n{inner_text}")
    
    if zones["outer_ring"]:
        outer_text = "\n".join(f"  • {r['text']}" for r in zones["outer_ring"])
        sections.append(f"BACKGROUND (may be tangential):\n{outer_text}")
    
    context = "\n\n".join(sections)
    return (
        f"{context}\n\n"
        f"QUESTION: {query}\n\n"
        f"INSTRUCTIONS: Answer primarily from PRIMARY EVIDENCE. "
        f"Use SUPPORTING CONTEXT to fill gaps. "
        f"Treat BACKGROUND as potentially unreliable. "
        f"Be concise and cite which evidence level supports each claim."
    )

# Demo: show both prompts side by side
query = "How efficient are solar panels?"
results = retrieve_top_k(query, k=7)
zones, _, _ = classify_zones_percentile(results)

print("═" * 70)
print("FLAT PROMPT (first 500 chars):")
print("═" * 70)
print(build_flat_prompt(query, results)[:500])
print()
print("═" * 70)
print("DARTBOARD PROMPT (first 500 chars):")
print("═" * 70)
print(build_dartboard_prompt(query, zones)[:500])

## 13 — The Complete Dartboard Retriever

Let's wrap everything into a single, clean function that takes a query and returns
zone-classified results with dynamically computed thresholds.

In [ ]:
def dartboard_retrieve(query, k=7, method="percentile", **kwargs):
    """
    Full dartboard retrieval pipeline:
      1. Embed query
      2. FAISS top-K search
      3. Classify into zones (dynamic thresholds)
      4. Return zones + metadata
    """
    # Step 1-2: retrieve
    results = retrieve_top_k(query, k=k)
    
    # Step 3: classify
    if method == "percentile":
        zones, t_bull, t_inner = classify_zones_percentile(
            results,
            bull_pct=kwargs.get("bull_pct", 80),
            inner_pct=kwargs.get("inner_pct", 40),
        )
    elif method == "stddev":
        zones, t_bull, t_inner = classify_zones_stddev(
            results,
            bull_k=kwargs.get("bull_k", 1.0),
            inner_k=kwargs.get("inner_k", -0.5),
        )
    elif method == "fixed":
        t_bull = kwargs.get("t_bull", 0.75)
        t_inner = kwargs.get("t_inner", 0.55)
        zones = classify_zones_fixed(results, t_bull, t_inner)
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Metadata
    meta = {
        "query": query,
        "method": method,
        "k": k,
        "t_bullseye": float(t_bull),
        "t_inner": float(t_inner),
        "counts": {z: len(v) for z, v in zones.items()},
    }
    return zones, meta

# Test
zones, meta = dartboard_retrieve("How does battery degradation work?")
print(f"Thresholds: T_bull={meta['t_bullseye']:.3f}, T_inner={meta['t_inner']:.3f}")
print(f"Zone counts: {meta['counts']}")
print()
for zone_name in ["bullseye", "inner_ring", "outer_ring"]:
    print(f"  {zone_name.upper()}:")
    for item in zones[zone_name]:
        print(f"    [{item['chunk_id']:>2}] {item['score']:.4f} — {item['text'][:55]}...")
    if not zones[zone_name]:
        print(f"    (empty)")

## 14 — Head-to-Head: Flat Top-K vs. Dartboard

Now the moment of truth. We run the **same queries** through both pipelines and compare:
1. What the LLM sees (prompt structure)
2. What the LLM generates (answer quality)

The hypothesis: dartboard prompts will produce **more focused, evidence-grounded** answers
because the LLM knows which information is most reliable.

In [ ]:
test_queries = [
    "What is the current efficiency record for perovskite-silicon tandem solar cells?",
    "Why do lithium-ion batteries degrade over time?",
    "How do carbon pricing mechanisms work in the EU?",
]

for q in test_queries:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("=" * 70)
    
    # Flat top-K
    flat_results = retrieve_top_k(q, k=7)
    flat_prompt = build_flat_prompt(q, flat_results)
    
    # Dartboard
    zones, meta = dartboard_retrieve(q, k=7)
    dart_prompt = build_dartboard_prompt(q, zones)
    
    print(f"\nFlat: all {len(flat_results)} chunks equal weight")
    print(f"Dartboard zones: {meta['counts']}")
    print(f"Thresholds: T_bull={meta['t_bullseye']:.3f}, T_inner={meta['t_inner']:.3f}")
    
    # Generate both answers
    print(f"\n--- FLAT ANSWER ---")
    flat_answer = generate(flat_prompt, max_new_tokens=200)
    print(flat_answer)
    
    print(f"\n--- DARTBOARD ANSWER ---")
    dart_answer = generate(dart_prompt, max_new_tokens=200)
    print(dart_answer)
    print()

## 15 — Token-Budget Allocation by Zone

Another way to implement dartboard weighting is to **control how much context window
each zone gets**. Instead of including all chunks, we allocate a token budget:

| Zone | Budget share | Rationale |
|---|---|---|
| Bullseye | 50% | Most important — give it the most room |
| Inner Ring | 35% | Useful supporting detail |
| Outer Ring | 15% | Only if there's space |

This prevents low-relevance chunks from crowding out high-relevance ones in
the limited context window.

In [ ]:
def allocate_token_budget(zones, total_tokens=1500, weights=None):
    """
    Allocate a token budget across zones and truncate chunks accordingly.
    Returns a new zones dict with possibly fewer/shorter chunks.
    """
    if weights is None:
        weights = {"bullseye": 0.50, "inner_ring": 0.35, "outer_ring": 0.15}
    
    budgeted = {}
    for zone_name in ["bullseye", "inner_ring", "outer_ring"]:
        budget = int(total_tokens * weights[zone_name])
        selected = []
        tokens_used = 0
        for item in zones[zone_name]:
            # Rough token estimate: ~4 chars per token
            chunk_tokens = len(item["text"]) // 4
            if tokens_used + chunk_tokens <= budget:
                selected.append(item)
                tokens_used += chunk_tokens
            elif tokens_used < budget:
                # Truncate to fit remaining budget
                remaining = (budget - tokens_used) * 4
                truncated = {**item, "text": item["text"][:remaining] + "…"}
                selected.append(truncated)
                tokens_used = budget
                break
            else:
                break
        budgeted[zone_name] = selected
    
    return budgeted

# Demo
query = "How efficient are solar panels?"
zones, meta = dartboard_retrieve(query, k=10)
budgeted = allocate_token_budget(zones, total_tokens=1000)

print(f"Query: '{query}'\n")
for zone_name in ["bullseye", "inner_ring", "outer_ring"]:
    orig_count = len(zones[zone_name])
    budg_count = len(budgeted[zone_name])
    print(f"  {zone_name.upper()}: {orig_count} → {budg_count} chunks after budget")
    for item in budgeted[zone_name]:
        print(f"    [{item['chunk_id']:>2}] {item['text'][:65]}...")
    print()

## 16 — Score-Weighted Context Construction

A more granular approach: instead of just *labelling* zones, we **weight each chunk's
contribution** by its similarity score. Bullseye chunks appear with full text;
outer ring chunks are summarised or condensed.

This is useful when you want continuous weighting rather than discrete zones.

In [ ]:
def build_weighted_prompt(query, results):
    """Build a prompt where each chunk gets space proportional to its score."""
    total_score = sum(r["score"] for r in results)
    max_chars = 3000  # total budget in characters
    
    sections = []
    for r in results:
        weight = r["score"] / total_score
        char_budget = int(max_chars * weight)
        text = r["text"][:char_budget]
        if len(r["text"]) > char_budget:
            text += "…"
        sections.append(f"[Relevance: {r['score']:.2f}] {text}")
    
    context = "\n\n".join(sections)
    return (
        f"Context (ordered by relevance, higher-scored passages are more reliable):\n"
        f"{context}\n\n"
        f"Question: {query}\n"
        f"Answer using the context, prioritising higher-relevance passages."
    )

# Demo
query = "What is the capacity factor of offshore wind farms?"
results = retrieve_top_k(query, k=7)
prompt = build_weighted_prompt(query, results)
print("Weighted prompt structure:")
for line in prompt.split("\n")[:12]:
    print(f"  {line[:90]}{'...' if len(line) > 90 else ''}")

## 17 — Complete Dartboard RAG Pipeline

Let's assemble everything into a single end-to-end function.

In [ ]:
def dartboard_rag(query, k=7, method="percentile", max_new_tokens=300, **kwargs):
    """
    Complete dartboard RAG pipeline:
      1. Retrieve top-K from FAISS
      2. Classify into zones with dynamic thresholds
      3. Apply token-budget allocation
      4. Build zone-aware prompt
      5. Generate answer with Qwen3-14B
    
    Returns: (answer, zones, metadata)
    """
    # Retrieve & classify
    zones, meta = dartboard_retrieve(query, k=k, method=method, **kwargs)
    
    # Budget allocation
    budgeted = allocate_token_budget(zones)
    
    # Build prompt
    prompt = build_dartboard_prompt(query, budgeted)
    
    # Generate
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    
    meta["prompt_length_chars"] = len(prompt)
    return answer, budgeted, meta


def flat_rag(query, k=7, max_new_tokens=300):
    """Standard flat RAG for comparison."""
    results = retrieve_top_k(query, k=k)
    prompt = build_flat_prompt(query, results)
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    return answer, results

print("Pipeline functions defined: dartboard_rag() and flat_rag()")

In [ ]:
# Run the full pipeline
query = "How do perovskite tandem solar cells achieve higher efficiency?"

answer, zones, meta = dartboard_rag(query)

print(f"Query: {query}\n")
print(f"Method: {meta['method']}")
print(f"Thresholds: T_bull={meta['t_bullseye']:.3f}, T_inner={meta['t_inner']:.3f}")
print(f"Zone counts: {meta['counts']}")
print(f"Prompt length: {meta['prompt_length_chars']} chars\n")
print(f"Answer:\n{answer}")

## 18 — Systematic Comparison: 5 Queries

We run five diverse queries through both pipelines and compare the results.
For each query we print the zone distribution and both answers.

In [ ]:
comparison_queries = [
    "What drives the cost reduction of solar energy?",
    "How do offshore wind farms compare to onshore in capacity factor?",
    "What are the main cathode chemistries for grid-scale batteries?",
    "What is a just transition in energy policy?",
    "How can solid-state batteries improve safety?",
]

for i, q in enumerate(comparison_queries, 1):
    print(f"\n{'='*70}")
    print(f"QUERY {i}: {q}")
    print(f"{'='*70}")
    
    # Dartboard
    d_answer, d_zones, d_meta = dartboard_rag(q, k=7)
    
    # Flat
    f_answer, f_results = flat_rag(q, k=7)
    
    print(f"\nDartboard zones: {d_meta['counts']}")
    print(f"Thresholds: T_bull={d_meta['t_bullseye']:.3f}, T_inner={d_meta['t_inner']:.3f}")
    
    print(f"\n--- FLAT ANSWER ---")
    print(f_answer[:300])
    
    print(f"\n--- DARTBOARD ANSWER ---")
    print(d_answer[:300])

## 19 — Tuning Zone Thresholds

The percentile and standard-deviation parameters control how aggressive the zoning is.
Let's explore the effect of different settings on zone distribution.

- **Tighter bullseye** (e.g., 90th percentile) → fewer bullseye chunks, higher precision
- **Looser bullseye** (e.g., 60th percentile) → more bullseye chunks, higher recall
- **Wider inner ring** (e.g., inner_pct=20) → more chunks get "supporting" status

In [ ]:
query = "How do feed-in tariffs support renewable energy?"
results = retrieve_top_k(query, k=10)

print(f"Query: '{query}'\n")
print(f"{'Bull_pct':>10} {'Inner_pct':>10} │ {'Bull':>5} {'Inner':>6} {'Outer':>6} │ T_bull  T_inner")
print("─" * 70)

for bull_pct in [90, 80, 70, 60]:
    for inner_pct in [50, 40, 30, 20]:
        z, tb, ti = classify_zones_percentile(results, bull_pct=bull_pct, inner_pct=inner_pct)
        b, i, o = len(z["bullseye"]), len(z["inner_ring"]), len(z["outer_ring"])
        print(f"{bull_pct:>10} {inner_pct:>10} │ {b:>5} {i:>6} {o:>6} │ {tb:.4f}  {ti:.4f}")

## 20 — Handling Edge Cases

What happens when the dartboard produces degenerate zones?

1. **Empty bullseye:** No chunk meets the bullseye threshold → fall back to flat top-K
   or use the single highest-scoring chunk as a "forced bullseye".

2. **Everything in bullseye:** All chunks score similarly (e.g., broad query) → the
   zoning adds no value, degrade gracefully to flat retrieval.

3. **Single result:** k=1 → one chunk, always bullseye. No zones needed.

A robust implementation handles all three gracefully.

In [ ]:
def dartboard_retrieve_robust(query, k=7, method="percentile", **kwargs):
    """
    Robust dartboard retrieval with edge-case handling.
    Falls back to flat retrieval when zoning is degenerate.
    """
    zones, meta = dartboard_retrieve(query, k=k, method=method, **kwargs)
    
    # Edge case 1: empty bullseye → force top result into bullseye
    if not zones["bullseye"] and (zones["inner_ring"] or zones["outer_ring"]):
        all_items = zones["inner_ring"] + zones["outer_ring"]
        best = max(all_items, key=lambda x: x["score"])
        best["zone"] = "bullseye"
        zones["bullseye"] = [best]
        # Remove from its original zone
        for z in ["inner_ring", "outer_ring"]:
            zones[z] = [r for r in zones[z] if r["chunk_id"] != best["chunk_id"]]
        meta["fallback"] = "forced_bullseye"
    
    # Edge case 2: everything in bullseye → zoning not useful
    total = sum(len(v) for v in zones.values())
    if zones["bullseye"] and len(zones["bullseye"]) == total:
        meta["fallback"] = "all_bullseye_flat"
    
    meta["counts"] = {z: len(v) for z, v in zones.items()}
    return zones, meta

# Test edge cases
print("=== Narrow query (likely good zoning) ===")
z, m = dartboard_retrieve_robust("Shockley-Queisser limit for solar cells")
print(f"  Counts: {m['counts']}, fallback: {m.get('fallback', 'none')}\n")

print("=== Broad query (may trigger edge case) ===")
z, m = dartboard_retrieve_robust("energy")
print(f"  Counts: {m['counts']}, fallback: {m.get('fallback', 'none')}\n")

print("=== k=1 ===")
z, m = dartboard_retrieve_robust("solar efficiency", k=1)
print(f"  Counts: {m['counts']}, fallback: {m.get('fallback', 'none')}")

## 21 — Measuring the "Spread" — When Does Dartboard Help?

The dartboard approach adds the most value when the **score spread** is large — i.e.,
there's a meaningful gap between the best and worst retrieved chunks.

When all scores are similar (low spread), zoning adds little value. We can measure this
with the **coefficient of variation** (CV = σ/μ) of the score distribution:

- **High CV (> 0.15):** Scores are spread out → dartboard helps
- **Low CV (< 0.05):** Scores are clustered → flat top-K is fine

In [ ]:
analysis_queries = [
    "What is the Shockley-Queisser limit for single-junction solar cells?",
    "How does the EU Emissions Trading System work?",
    "Tell me about energy",
    "What is the cycle life of lithium iron phosphate batteries?",
    "Offshore wind capacity factor",
    "How are wind turbine blades recycled?",
]

print(f"{'Query':<55} {'CV':>6} {'Spread':>7} {'Dartboard?':>12}")
print("─" * 85)

for q in analysis_queries:
    res = retrieve_top_k(q, k=7)
    scores = np.array([r["score"] for r in res])
    cv = scores.std() / scores.mean() if scores.mean() > 0 else 0
    spread = scores.max() - scores.min()
    recommendation = "✓ YES" if cv > 0.10 else "~ maybe" if cv > 0.05 else "✗ flat OK"
    label = q[:53] + ".." if len(q) > 53 else q
    print(f"{label:<55} {cv:>6.3f} {spread:>7.3f} {recommendation:>12}")

## 22 — Adaptive Pipeline: Auto-Select Flat or Dartboard

We can build a pipeline that **automatically decides** whether to use dartboard or flat
retrieval based on the score spread. If the CV is below a threshold, we fall back to flat
retrieval — no unnecessary complexity.

In [ ]:
def adaptive_rag(query, k=7, cv_threshold=0.08, max_new_tokens=300):
    """
    Adaptive RAG: uses dartboard when score spread is high, flat otherwise.
    Returns (answer, method_used, metadata).
    """
    results = retrieve_top_k(query, k=k)
    scores = np.array([r["score"] for r in results])
    cv = scores.std() / scores.mean() if scores.mean() > 0 else 0
    
    if cv >= cv_threshold:
        # Dartboard
        zones, meta = dartboard_retrieve(query, k=k)
        budgeted = allocate_token_budget(zones)
        prompt = build_dartboard_prompt(query, budgeted)
        method_used = "dartboard"
    else:
        # Flat
        prompt = build_flat_prompt(query, results)
        meta = {"cv": cv}
        method_used = "flat"
    
    answer = generate(prompt, max_new_tokens=max_new_tokens)
    meta["cv"] = float(cv)
    meta["method_selected"] = method_used
    return answer, method_used, meta

# Test on two contrasting queries
for q in ["What is the Shockley-Queisser limit?", "Tell me about energy"]:
    answer, method, meta = adaptive_rag(q)
    print(f"Query: '{q}'")
    print(f"  CV={meta['cv']:.3f} → method={method}")
    print(f"  Answer: {answer[:150]}...\n")

## 23 — Synthesis: When to Use the Dartboard Approach

| Scenario | Recommendation |
|---|---|
| **Specific factual questions** with a clear correct answer | ✅ Dartboard — bullseye isolates the answer |
| **Broad exploratory questions** ("tell me about X") | ⚠️ Maybe — zones may be empty or unhelpful |
| **Small K** (1–3 results) | ❌ Little value — not enough results to zone |
| **Large K** (10–20 results) | ✅ Dartboard shines — prevents noise dilution |
| **Uniform embeddings** (scores clustered) | ❌ Zones are meaningless when scores are similar |
| **Diverse corpus** (multiple topics) | ✅ Clear score separation makes zoning effective |

### Threshold Tuning Rules of Thumb

1. **Start with percentile-based** (80th/40th) — it adapts automatically.
2. If you know your embedding model well, switch to **std-dev** (μ+σ / μ−0.5σ).
3. Use **CV-based adaptive selection** in production to avoid unnecessary overhead.
4. **Monitor bullseye occupancy** — if most queries have 0 or 1 bullseye chunks,
   lower the threshold.

In [ ]:
# Final demonstration: the full pipeline in action
print("╔══════════════════════════════════════════════════════════════╗")
print("║         DARTBOARD RAG — COMPLETE PIPELINE DEMO             ║")
print("╚══════════════════════════════════════════════════════════════╝\n")

demo_query = "What are the advantages of sodium-ion batteries over lithium-ion?"

# Step 1: Retrieve & classify
zones, meta = dartboard_retrieve_robust(demo_query, k=8)
print(f"Query: {demo_query}")
print(f"Method: {meta['method']}, K={meta['k']}")
print(f"Thresholds: T_bull={meta['t_bullseye']:.3f}, T_inner={meta['t_inner']:.3f}")
print(f"Zone distribution: {meta['counts']}")
print(f"Fallback: {meta.get('fallback', 'none')}\n")

# Step 2: Show zone contents
for zone_name in ["bullseye", "inner_ring", "outer_ring"]:
    emoji = {"bullseye": "🎯", "inner_ring": "🔵", "outer_ring": "⚪"}[zone_name]
    print(f"{emoji} {zone_name.upper()}:")
    for item in zones[zone_name]:
        print(f"   [{item['chunk_id']:>2}] sim={item['score']:.4f} — {item['text'][:60]}...")
    if not zones[zone_name]:
        print(f"   (empty)")
    print()

# Step 3: Budget & generate
budgeted = allocate_token_budget(zones)
prompt = build_dartboard_prompt(demo_query, budgeted)
answer = generate(prompt, max_new_tokens=300)

print("GENERATED ANSWER:")
print("─" * 60)
print(answer)

## 24 — Key Takeaways

1. **Standard top-K is a blunt instrument.** It ignores the natural structure in
   similarity scores. The dartboard approach exploits that structure.

2. **Zones are cheap.** Classifying chunks into bullseye/inner/outer adds negligible
   latency — it's just comparing scores against thresholds.

3. **Dynamic thresholds are essential.** Fixed thresholds break on queries with
   different score distributions. Use percentile or std-dev based methods.

4. **Zone-aware prompting is the main win.** The LLM benefits from knowing which
   evidence is strong and which is weak. Structured prompts yield more focused answers.

5. **Not always necessary.** When scores are clustered (low CV), zones add no value.
   The adaptive pipeline handles this automatically.

6. **Complements other techniques.** Dartboard works alongside reranking, query
   expansion, HyDE, and other RAG improvements — it's about what you do *after*
   retrieval, not how you retrieve.

**Next steps:** Try combining dartboard retrieval with cross-encoder reranking
(rerank first, then zone-classify the reranked results) for even better precision.